# 01. Data Understanding

# Load Dataset and Check Dimensions

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

data_path = "data/Dataset/Parquet/Combined Data/Combined_dataset.parquet"
out_dir = "data/GeneratedOutput/01.EDA"
if not os.path.exists(data_path):
    data_path = os.path.join("..", "..", data_path)
    out_dir = os.path.join("..", "..", out_dir)
os.makedirs(out_dir, exist_ok=True)

df = pd.read_parquet(data_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()

# Dataset Schema and Data Types

In [ ]:
schema_df = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str),
    "Non-Null Count": df.notnull().sum(),
    "Null %": (df.isnull().sum() / len(df) * 100).round(2)
}).reset_index(drop=True)
schema_df

# Country Distribution

In [ ]:
country_counts = df["Country"].value_counts()
country_pcts = (country_counts / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=country_counts.values, y=country_counts.index, palette="Blues_r", hue=country_counts.index, legend=False, ax=ax)
ax.set_xscale("log")
ax.set_title("Record Count by Country (Log Scale)")
ax.set_xlabel("Number of Records (Logarithmic)")
for i, count in enumerate(country_counts.values):
    ax.text(count * 1.15, i, f"{count:,} ({country_pcts.iloc[i]}%)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

pd.DataFrame({"Records": country_counts, "Share (%)": country_pcts})

# Waterbody Types and Monitoring Stations

In [ ]:
wb_counts = df["Waterbody Type"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=wb_counts.values, y=wb_counts.index, palette="crest_r", hue=wb_counts.index, legend=False, ax=ax)
ax.set_title("Distribution of Surface Waterbody Types")
ax.set_xlabel("Number of Measurements")
for i, count in enumerate(wb_counts.values):
    ax.text(count + 15000, i, f"{count:,} ({(count/len(df)*100):.1f}%)", va="center", fontsize=9)
plt.xlim(0, wb_counts.max() * 1.15)
plt.tight_layout()
plt.show()

print(f"Total Unique Monitoring Stations: {df["Area"].nunique():,}")

# Temporal Range of Observations

In [ ]:
df["Year"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce").dt.year
year_counts = df["Year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(year_counts.index, year_counts.values, color="#3b82f6", alpha=0.3)
ax.plot(year_counts.index, year_counts.values, color="#1d4ed8", lw=2)
ax.set_title("Temporal Sampling Frequency: 1940 to 2023")
ax.set_xlabel("Year")
ax.set_ylabel("Annual Measurements")
ax.axvline(2000, color="red", linestyle="--", alpha=0.7, label="Year 2000")
ax.legend()
plt.tight_layout()
plt.show()

summary_profile = pd.DataFrame({
    "Metric": ["Total Rows", "Total Columns", "Total Unique Stations", "Date Start", "Date End", "Top Country", "Top Waterbody"],
    "Value": [
        f"{len(df):,}",
        f"{len(df.columns)}",
        f"{df["Area"].nunique():,}",
        str(df["Date"].min()),
        str(df["Date"].max()),
        f"{df["Country"].value_counts().index[0]} ({df["Country"].value_counts().iloc[0]:,})",
        f"{df["Waterbody Type"].value_counts().index[0]} ({df["Waterbody Type"].value_counts().iloc[0]:,})"
    ]
})
summary_profile.to_parquet(os.path.join(out_dir, "01_dataset_summary_profile.parquet"), index=False, compression="snappy")

# Sensor Missingness Rates

In [ ]:
chemical_cols = [
    "Ammonia (mg/l)", "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)", "Orthophosphate (mg/l)",
    "pH (ph units)", "Temperature (cel)", "Nitrogen (mg/l)", "Nitrate (mg/l)"
]

null_counts = df[chemical_cols].isnull().sum()
null_pcts = (null_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Parameter": chemical_cols,
    "Missing Count": null_counts.values,
    "Missing %": null_pcts.values,
    "Present Count": (len(df) - null_counts).values
}).sort_values("Missing %", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=missing_df, x="Missing %", y="Parameter", palette="mako_r", hue="Parameter", legend=False, ax=ax)
ax.set_title("Percentage of Missing Values per Chemical Parameter")
ax.set_xlim(0, 40)
for i, pct in enumerate(missing_df["Missing %"]):
    ax.text(pct + 0.5, i, f"{pct:.1f}% ({missing_df["Missing Count"].iloc[i]:,})", va="center", fontsize=9)
plt.tight_layout()
plt.show()

missing_df

# Missingness Patterns Across Countries

In [ ]:
country_missing = df.groupby("Country")[chemical_cols].apply(lambda g: (g.isnull().mean() * 100).round(1))

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(country_missing, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Missing %"}, linewidths=0.5, ax=ax)
ax.set_title("Sensor Missingness Rate (%) Across Countries")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

country_missing.reset_index().to_parquet(os.path.join(out_dir, "02_missingness_patterns.parquet"), index=False, compression="snappy")

# Sensor Co-Occurrence and Complete Cases

In [ ]:
presence_matrix = df[chemical_cols].notnull().astype(int)
co_occurrence = presence_matrix.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(co_occurrence, dtype=bool))
sns.heatmap(co_occurrence, mask=mask, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Sensor Co-Occurrence Correlation")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

complete_count = df[chemical_cols].notnull().all(axis=1).sum()
print(f"Complete-Case Records (All 8 Sensors Active): {complete_count:,} ({complete_count/len(df)*100:.2f}%)")

# Numerical Summary Statistics

In [ ]:
df[chemical_cols].describe().round(3)